## 2d
- id lang + translate > fit into pipe
- test sentiment
- test commitment
- test many reports (1 click pipeline!)
- scores 
    - specificity: pdf-wide or add highest
    - Validate scoring by inspecting chunks
    - calc talk-score

### more ideas
- CLIMATE_CONFIDENCE_THRESHOLD = 0.7  # Only keep chunks with >70% climate confidence
- 

In [ ]:
"""
Optimized pipeline for large-scale climate report analysis
Features:
- Pre-filtering with climate detector
- Batch processing
- Caching preprocessed data
- Multi-stage analysis
- Multi-language support with translation
"""

import json
from pathlib import Path
from typing import List, Dict, Tuple
import pdfplumber
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline
from tqdm.auto import tqdm
import langid

class ClimateReportAnalyzer:
    # Configuration constants
    MIN_CHUNK_LENGTH = 200  # Minimum characters for a chunk to be analyzed
    BATCH_SIZE = 48         # Number of chunks to process in parallel

    def __init__(self, cache_dir="cache", use_cache=True):
        """
        Initialize analyzer with configuration

        Args:
            cache_dir: Directory for caching results
            use_cache: Use cached text extractions and results
        """
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(exist_ok=True)
        self.use_cache = use_cache

        self.device = 0 if torch.cuda.is_available() else -1
        # print(f"Using device: {'GPU' if self.device == 0 else 'CPU'}")
        # print(f"Cache enabled: {use_cache}")

        self.models = {}
        self.translators = {}

        # Helsinki-NLP translation models for common languages
        self.supported_languages = {
            'de': 'Helsinki-NLP/opus-mt-de-en',  # German
            'fr': 'Helsinki-NLP/opus-mt-fr-en',  # French
            'es': 'Helsinki-NLP/opus-mt-es-en',  # Spanish
            'it': 'Helsinki-NLP/opus-mt-it-en',  # Italian
            'pt': 'Helsinki-NLP/opus-mt-tc-big-pt-en',  # Portuguese
            'nl': 'Helsinki-NLP/opus-mt-nl-en',  # Dutch
            'pl': 'Helsinki-NLP/opus-mt-pl-en',  # Polish
            'sv': 'Helsinki-NLP/opus-mt-sv-en',  # Swedish
            'da': 'Helsinki-NLP/opus-mt-da-en',  # Danish
            'fi': 'Helsinki-NLP/opus-mt-fi-en',  # Finnish
            'no': 'Helsinki-NLP/opus-mt-no-en',  # Norwegian
            'ru': 'Helsinki-NLP/opus-mt-ru-en',  # Russian
            'zh': 'Helsinki-NLP/opus-mt-zh-en',  # Chinese
            'ja': 'Helsinki-NLP/opus-mt-ja-en',  # Japanese
            'ko': 'Helsinki-NLP/opus-mt-ko-en',  # Korean
        }

    def load_model(self, model_name: str, task_name: str):
        """Load and cache a model"""
        if task_name not in self.models:
            print(f"Loading {task_name} model...")
            model = AutoModelForSequenceClassification.from_pretrained(model_name)
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.models[task_name] = pipeline(
                "text-classification",
                model=model,
                tokenizer=tokenizer,
                device=self.device
            )
        return self.models[task_name]

    def extract_text_from_pdf(self, pdf_path: str) -> str:
        """Extract text from PDF with caching"""
        cache_file = self.cache_dir / f"{Path(pdf_path).stem}_text.txt"

        if self.use_cache and cache_file.exists():
            print(f"Loading cached text from {cache_file}")
            return cache_file.read_text(encoding='utf-8')

        print(f"Extracting text from {pdf_path}...")
        text = ""
        with pdfplumber.open(pdf_path) as pdf:
            for page in tqdm(pdf.pages, desc="Reading pages"):
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"

        if self.use_cache:
            cache_file.write_text(text, encoding='utf-8')
            print(f"Cached text to {cache_file}")

        return text

    def chunk_text_by_paragraphs(self, text: str, max_tokens=400) -> List[str]:
        """
        Split by paragraphs, merge small ones, split large ones, filter really small (<MIN_CHUNK_LENGTH)
        Note: no need overlap in classification/sentiment/topic detect/commitment (would need for NER, summarization)

        Args:
            max_tokens: Maximum tokens per chunk (default 400, model max is 512)

        Returns:
            List of text chunks, each roughly paragraph-sized
        """
        # Rough estimate: 1 token ≈ 4 characters
        MAX_CHARS = max_tokens * 4  # 400 tokens ≈ 1600 chars

        # Step 1: Split by double newline (paragraphs)
        paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]

        chunks = []
        current_chunk = ""

        for para in paragraphs:
            para_size = len(para)
            current_size = len(current_chunk)

            # Case A: Paragraph is way too big → split it by sentences
            if para_size > MAX_CHARS:
                # Save current chunk first
                if current_chunk:
                    chunks.append(current_chunk.strip())
                    current_chunk = ""

                # Split large paragraph into sentences
                sentences = para.split('. ')
                for sent in sentences:
                    if current_size + len(sent) < MAX_CHARS:
                        current_chunk += sent + ". "
                        current_size = len(current_chunk)
                    else:
                        if current_chunk:
                            chunks.append(current_chunk.strip())
                        current_chunk = sent + ". "
                        current_size = len(current_chunk)

            # Case B: Adding this paragraph would exceed limit → save & start new
            elif current_size + para_size > MAX_CHARS:
                if current_chunk:
                    chunks.append(current_chunk.strip())
                current_chunk = para

            # Case C: Small paragraph → accumulate
            else:
                current_chunk += ("\n\n" if current_chunk else "") + para

        # Don't forget last chunk
        if current_chunk:
            chunks.append(current_chunk.strip())

        # Filter very short chunks
        return [c for c in chunks if len(c) > self.MIN_CHUNK_LENGTH]

    def detect_language(self, text: str) -> str:
        """
        Detect language of text using langid

        Args:
            text: Text sample to detect language from

        Returns:
            ISO 639-1 language code (e.g., 'en', 'de', 'es')
        """
        try:
            # Use first 5000 chars for detection (more reliable)
            sample = text[:5000]
            lang, log_likelyhood = langid.classify(sample)
            print(f"Language detection: {lang}") # (log_likelyhood: {log_likelyhood:.3f})")
            return lang
        except Exception as e:
            print(f"Language detection failed: {e}. Defaulting to 'en'")
            return 'en'

    def load_translator(self, lang: str):
        """Lazy load translation model for given language"""
        if lang not in self.supported_languages:
            raise ValueError(f"Translation not supported for language: {lang}. "
                           f"Supported: {list(self.supported_languages.keys())}")

        if lang not in self.translators:
            print(f"Loading translation model for {lang}→en...")
            model_name = self.supported_languages[lang]
            self.translators[lang] = pipeline(
                "translation",
                model=model_name,
                device=self.device,
                max_length=512
            )

        return self.translators[lang]

    def translate_to_english(self, chunks: List[str], lang: str,
                            batch_size=8) -> Tuple[List[str], str]:
        """
        Translate chunks from source language to English
        Handles chunks longer than 512 tokens by truncating them

        Args:
            chunks: List of text chunks in source language
            lang: Source language code
            batch_size: Batch size for translation
                > Smaller batch for translation (memory intensive)

        Returns:
            tuple: (translated_chunks, cache_suffix)
        """

        cache_suffix = f"_translated_{lang}_en"

        translator = self.load_translator(lang)
        translated_chunks = []

        print(f"\nTranslating {len(chunks)} chunks from {lang} to English...")

        for i in tqdm(range(0, len(chunks), batch_size), desc="Translating"):
            batch = chunks[i:i+batch_size]
            try:
                # Translation models have 512 token limit
                # Truncate long chunks to ~450 tokens (1800 chars) to be safe
                truncated_batch = [chunk[:1800] for chunk in batch]

                results = translator(truncated_batch,
                                   batch_size=batch_size,
                                   truncation=True,
                                   max_length=600)  # Allow longer output than input

                for result in results:
                    # Extract translated text
                    if isinstance(result, dict):
                        translated_text = result.get('translation_text', '')
                    elif isinstance(result, list) and len(result) > 0:
                        translated_text = result[0].get('translation_text', '')
                    else:
                        translated_text = str(result)

                    print(f"Result type: {type(result)}")
                    print(f"Result content: {result}")
                    print(f"Extracted text: {translated_text[:100]}")

                    translated_chunks.append(translated_text)

            except Exception as e:
                print(f"\nTranslation error in batch {i//batch_size}: {e}")
                # Fallback: keep original text for failed translations
                translated_chunks.extend(batch)
                continue

        print(f"Translation complete: {len(translated_chunks)} chunks")
        return translated_chunks, cache_suffix

    def filter_climate_chunks(self, chunks: List[str], batch_size=None) -> List[Dict]:
        """Filter chunks to keep only climate-related ones"""
        if batch_size is None:
            batch_size = self.BATCH_SIZE

        detector = self.load_model(
            "climatebert/distilroberta-base-climate-detector",
            "detector"
        )

        print(f"\nFiltering {len(chunks)} chunks for climate content...")
        climate_chunks = []

        # Process in batches for MUCH better performance (10-20x speedup)
        for i in tqdm(range(0, len(chunks), batch_size), desc="Detecting climate content"):
            batch = chunks[i:i+batch_size]
            try:
                # KEY: Pass batch_size to enable batching in the pipeline
                results = detector(batch, truncation=True, max_length=512, batch_size=batch_size)

                for chunk, result in zip(batch, results):
                    # Keep if labeled as "climate" or similar
                    if result['label'].lower() in ['climate', 'yes', 'climate-related']:
                        climate_chunks.append({
                            'text': chunk,
                            'detector_score': result['score']
                        })
            except Exception as e:
                print(f"Error in batch {i}: {e}")
                continue

        print(f"Kept {len(climate_chunks)} climate-related chunks ({len(climate_chunks)/len(chunks)*100:.1f}%)")
        return climate_chunks

    def analyze_specificity(self, chunks: List[Dict], batch_size=None) -> List[Dict]:
        """Analyze climate specificity with batching for performance"""
        if batch_size is None:
            batch_size = self.BATCH_SIZE

        specificity = self.load_model(
            "climatebert/distilroberta-base-climate-specificity",
            "specificity"
        )

        print(f"\nAnalyzing specificity for {len(chunks)} chunks...")

        for i in tqdm(range(0, len(chunks), batch_size), desc="Analyzing specificity"):
            batch_chunks = chunks[i:i+batch_size]
            batch_texts = [c['text'] for c in batch_chunks]

            try:
                # BATCHED: 10-20x faster than processing one-by-one
                results = specificity(batch_texts, truncation=True, max_length=512, batch_size=batch_size)

                for chunk, result in zip(batch_chunks, results):
                    chunk['specificity_label'] = result['label']
                    chunk['specificity_score'] = result['score']
            except Exception as e:
                print(f"Error in batch {i}: {e}")
                continue

        return chunks

    def analyze_sentiment(self, chunks: List[Dict], batch_size=None) -> List[Dict]:
        """Analyze climate sentiment (opportunity/neutral/risk)"""
        if batch_size is None:
            batch_size = self.BATCH_SIZE

        sentiment = self.load_model(
            "climatebert/distilroberta-base-climate-sentiment",
            "sentiment"
        )

        print(f"\nAnalyzing sentiment for {len(chunks)} chunks...")

        for i in tqdm(range(0, len(chunks), batch_size), desc="Analyzing sentiment"):
            batch_chunks = chunks[i:i+batch_size]
            batch_texts = [c['text'] for c in batch_chunks]

            try:
                results = sentiment(batch_texts, truncation=True, max_length=512, batch_size=batch_size)

                for chunk, result in zip(batch_chunks, results):
                    chunk['sentiment_label'] = result['label']
                    chunk['sentiment_score'] = result['score']
            except Exception as e:
                print(f"Error in batch {i}: {e}")
                continue

        return chunks

    def analyze_commitments(self, chunks: List[Dict], batch_size=None) -> List[Dict]:
        """Detect climate commitments and actions"""
        if batch_size is None:
            batch_size = self.BATCH_SIZE

        commitment = self.load_model(
            "climatebert/distilroberta-base-climate-commitment",
            "commitment"
        )

        print(f"\nAnalyzing commitments for {len(chunks)} chunks...")

        for i in tqdm(range(0, len(chunks), batch_size), desc="Analyzing commitments"):
            batch_chunks = chunks[i:i+batch_size]
            batch_texts = [c['text'] for c in batch_chunks]

            try:
                results = commitment(batch_texts, truncation=True, max_length=512, batch_size=batch_size)

                for chunk, result in zip(batch_chunks, results):
                    chunk['commitment_label'] = result['label']
                    chunk['commitment_score'] = result['score']
            except Exception as e:
                print(f"Error in batch {i}: {e}")
                continue

        return chunks

    def extract_and_prepare_text(self, pdf_path: str, translate=False):
        """
        Step 1: Extract text, detect language, optionally translate

        Args:
            pdf_path: Path to PDF file
            translate: Whether to translate non-English text

        Returns:
            tuple: (chunks, lang, cache_suffix)
        """
        print(f"\n{'='*80}")
        print(f"STEP 1: Extract & Prepare Text")
        print(f"{'='*80}")

        pdf_stem = Path(pdf_path).stem

        # Extract text
        text = self.extract_text_from_pdf(pdf_path)
        print(f"Total text length: {len(text)} characters")

        # Detect language
        lang_cache = self.cache_dir / f"{pdf_stem}_lang.txt"

        if self.use_cache and lang_cache.exists():
            lang = lang_cache.read_text(encoding='utf-8').strip()
            print(f"Loaded cached language: {lang}")
        else:
            lang = self.detect_language(text)
            if self.use_cache:
                lang_cache.write_text(lang, encoding='utf-8')

        # Chunk text
        chunks = self.chunk_text_by_paragraphs(text)
        print(f"Created {len(chunks)} chunks")

        # Translate if needed
        cache_suffix = ""
        if lang != 'en' and translate:
            if lang not in self.supported_languages:
                print(f"⚠️  Translation not supported for language: {lang}")
                print(f"   Supported languages: {list(self.supported_languages.keys())}")
                print(f"   Proceeding with original text (may affect analysis accuracy)")
                cache_suffix = f"_unsupported_{lang}"
            else:
                orig_chunks_cache = self.cache_dir / f"{pdf_stem}_original_chunks.json"
                if not orig_chunks_cache.exists():
                    with open(orig_chunks_cache, 'w', encoding='utf-8') as f:
                        json.dump(chunks, f, ensure_ascii=False, indent=2)

                trans_cache = self.cache_dir / f"{pdf_stem}_translated_{lang}_en.json"

                if self.use_cache and trans_cache.exists():
                    print(f"Loading cached translation from {trans_cache}")
                    with open(trans_cache, 'r', encoding='utf-8') as f:
                        chunks = json.load(f)
                    cache_suffix = f"_translated_{lang}_en"
                else:
                    chunks, cache_suffix = self.translate_to_english(chunks, lang)
                    print(f"chunks: {chunks}!!!!!\n\n\n\n")
                    if self.use_cache:
                        with open(trans_cache, 'w', encoding='utf-8') as f:
                            json.dump(chunks, f, ensure_ascii=False, indent=2)
                        print(f"Cached translation to {trans_cache}")

        return chunks, lang, cache_suffix

    def calculate_report_score(self, analyzed_chunks: List[Dict]) -> Dict:
        """Calculate overall scores for the report"""
        # Correct labels for climate-specificity model
        score_map = {
            "specific": 1.0,
            "non-specific": 0.0,
        }

        specificity_scores = []
        for chunk in analyzed_chunks:
            label = chunk.get('specificity_label', '').lower()
            confidence = chunk.get('specificity_score', 0.5)

            # Map label to score
            base_score = score_map.get(label, 0.5)  # Default 0.5 if unknown
            weighted_score = base_score * confidence
            specificity_scores.append(weighted_score)

        overall_score = sum(specificity_scores) / len(specificity_scores) if specificity_scores else 0

        # Count labels
        label_dist = {}
        for chunk in analyzed_chunks:
            label = chunk.get('specificity_label', 'unknown')
            label_dist[label] = label_dist.get(label, 0) + 1

        return {
            'overall_specificity': overall_score,
            'num_chunks_analyzed': len(analyzed_chunks),
            'num_climate_chunks': len([c for c in analyzed_chunks if c.get('specificity_label') == 'specific']),
            'label_distribution': label_dist,
            'specificity_scores': specificity_scores
        }

    def inspect_chunks_interactive(self, pdf_path: str, mode="random", n=3,
                                  min_confidence=0.0, max_confidence=1.0,
                                  label_filter=None):
        """
        Advanced chunk inspection for manual evaluation

        Args:
            pdf_path: Path to PDF
            mode: "random", "high_conf", "low_conf", "high_score", "low_score"
            n: Number of chunks to show
            min_confidence: Minimum confidence threshold
            max_confidence: Maximum confidence threshold
            label_filter: Filter by label (e.g., "specific", "opportunity")
        """
        import random

        # Load results
        results_file = self.cache_dir / f"{Path(pdf_path).stem}_results.json"
        if not results_file.exists():
            print(f"No results found. Run process() first.")
            return

        with open(results_file, encoding='utf-8') as f:
            data = json.load(f)

        chunks = data.get('sample_chunks', [])
        if not chunks:
            print("No chunks available. Re-run analysis.")
            return

        # Filter by label if specified
        if label_filter:
            chunks = [c for c in chunks if label_filter.lower() in
                     str(c.get('specificity_label', '')).lower() or
                     label_filter.lower() in str(c.get('sentiment_label', '')).lower()]

        # Filter by confidence range
        chunks = [c for c in chunks if min_confidence <= c.get('specificity_score', 0.5) <= max_confidence]

        if not chunks:
            print(f"No chunks matching criteria (label={label_filter}, conf={min_confidence}-{max_confidence})")
            return

        # Select chunks based on mode
        if mode == "random":
            selected = random.sample(chunks, min(n, len(chunks)))
        elif mode == "high_conf":
            sorted_chunks = sorted(chunks, key=lambda x: x.get('specificity_score', 0), reverse=True)
            selected = sorted_chunks[:n]
        elif mode == "low_conf":
            sorted_chunks = sorted(chunks, key=lambda x: x.get('specificity_score', 1))
            selected = sorted_chunks[:n]
        else:
            selected = chunks[:n]

        # Display
        print(f"\n{'='*80}")
        print(f"CHUNK INSPECTION - Mode: {mode}")
        print(f"PDF: {Path(pdf_path).name}")
        print(f"Showing {len(selected)} of {len(chunks)} chunks")
        print(f"{'='*80}\n")

        for i, chunk in enumerate(selected, 1):
            print(f"{'─'*80}")
            print(f"Chunk #{i}")
            print(f"{'─'*80}")

            # Show all scores
            spec_label = chunk.get('specificity_label', 'N/A')
            spec_conf = chunk.get('specificity_score', 0)
            print(f"Specificity: {spec_label} (confidence: {spec_conf:.3f})")

            if 'sentiment_label' in chunk:
                sent_label = chunk.get('sentiment_label')
                sent_conf = chunk.get('sentiment_score', 0)
                print(f"Sentiment:   {sent_label} (confidence: {sent_conf:.3f})")

            if 'commitment_label' in chunk:
                commit_label = chunk.get('commitment_label')
                commit_conf = chunk.get('commitment_score', 0)
                print(f"Commitment:  {commit_label} (confidence: {commit_conf:.3f})")

            if 'detector_score' in chunk:
                det_conf = chunk.get('detector_score')
                print(f"Climate relevance: {det_conf:.3f}")

            text = chunk.get('text', '')
            print(f"\nText ({len(text)} chars):")
            print(f"{text[:500]}{'...' if len(text) > 500 else ''}\n")

    def check_translations(self, reports_dir: str = None):
        """
        Check language detection and translation status for all cached PDFs

        Args:
            reports_dir: Directory to scan (optional, uses cache if None)
        """
        print(f"\n{'='*80}")
        print(f"TRANSLATION STATUS CHECK")
        print(f"{'='*80}\n")

        # Find all result files
        result_files = list(self.cache_dir.glob("*_results.json"))

        if not result_files:
            print("No cached results found. Process some PDFs first.")
            return

        stats = {
            'en': 0,
            'translated': 0,
            'unsupported': 0,
            'by_language': {}
        }

        print(f"{'PDF':<40} {'Lang':<6} {'Status':<30}")
        print(f"{'-'*80}")

        for result_file in sorted(result_files):
            try:
                with open(result_file, 'r', encoding='utf-8') as f:
                    data = json.load(f)

                pdf_name = Path(data.get('pdf_path', result_file.stem)).name[:38]
                lang = data.get('language', 'unknown')
                translation = data.get('translation', 'unknown')

                # Update stats
                if lang not in stats['by_language']:
                    stats['by_language'][lang] = 0
                stats['by_language'][lang] += 1

                if lang == 'en':
                    stats['en'] += 1
                    status_icon = "✓"
                elif 'translated' in translation:
                    stats['translated'] += 1
                    status_icon = "🔄"
                elif 'unsupported' in translation:
                    stats['unsupported'] += 1
                    status_icon = "⚠️ "
                else:
                    status_icon = "?"

                print(f"{pdf_name:<40} {lang:<6} {status_icon} {translation}")

            except Exception as e:
                print(f"{result_file.name:<40} ERROR: {e}")

        # Summary
        print(f"\n{'-'*80}")
        print(f"SUMMARY")
        print(f"{'-'*80}")
        total = len(result_files)
        print(f"Total PDFs processed: {total}")
        print(f"  ✓ English (no translation): {stats['en']} ({stats['en']/total*100:.1f}%)")
        print(f"  🔄 Translated: {stats['translated']} ({stats['translated']/total*100:.1f}%)")
        print(f"  ⚠️  Unsupported: {stats['unsupported']} ({stats['unsupported']/total*100:.1f}%)")

        print(f"\nLanguage breakdown:")
        for lang, count in sorted(stats['by_language'].items(), key=lambda x: x[1], reverse=True):
            print(f"  {lang}: {count}")

    def inspect_translation(self, pdf_path: str, n_samples=3):
        """
        Show original and translated text side-by-side for validation

        Args:
            pdf_path: Path to PDF
            n_samples: Number of chunk pairs to show
        """
        pdf_stem = Path(pdf_path).stem

        # Load original text and language
        orig_cache = self.cache_dir / f"{pdf_stem}_text.txt"
        lang_file = self.cache_dir / f"{pdf_stem}_lang.txt"

        if not orig_cache.exists() or not lang_file.exists():
            print(f"No cached data found for {pdf_path}")
            return

        orig_text = orig_cache.read_text(encoding='utf-8')
        lang = lang_file.read_text(encoding='utf-8').strip()

        if lang == 'en':
            print(f"PDF is in English, no translation needed.")
            return

        # Load translated chunks
        trans_cache = self.cache_dir / f"{pdf_stem}_translated_{lang}_en.json"
        if not trans_cache.exists():
            print(f"No translation found. Run process() with auto_translate=True first.")
            return

        with open(trans_cache, 'r', encoding='utf-8') as f:
            translated_chunks = json.load(f)

        # Get original chunks
        orig_chunks = self.chunk_text_by_paragraphs(orig_text)

        print(f"\n{'='*80}")
        print(f"TRANSLATION INSPECTION: {Path(pdf_path).name}")
        print(f"Language: {lang} → en")
        print(f"{'='*80}\n")

        # Show random samples
        import random
        indices = random.sample(range(min(len(orig_chunks), len(translated_chunks))),
                               min(n_samples, len(orig_chunks), len(translated_chunks)))

        for i, idx in enumerate(indices, 1):
            orig = orig_chunks[idx]
            trans = translated_chunks[idx] if idx < len(translated_chunks) else "N/A"

            print(f"{'─'*80}")
            print(f"Sample {i} (chunk #{idx})")
            print(f"{'─'*80}")
            print(f"\nORIGINAL ({lang}):")
            print(f"{orig[:400]}{'...' if len(orig) > 400 else ''}")
            print(f"\nTRANSLATED (en):")
            print(f"{trans[:400]}{'...' if len(trans) > 400 else ''}\n")



In [ ]:
# Initialize analyzer
analyzer = ClimateReportAnalyzer(use_cache=False)

pdf_path = "data/reports/Dillinger/012_2015_annual_report.pdf"


# Step 1: Extract and prepare text (with translation)
chunks, lang, cache_suffix = analyzer.extract_and_prepare_text( pdf_path, translate=True )
print(f"\n✅ Step 1 complete: {len(chunks)} chunks ready, language: {lang}")





STEP 1: Extract & Prepare Text
Extracting text from data/reports/Dillinger/012_2015_annual_report.pdf...


Reading pages:   0%|          | 0/70 [00:00<?, ?it/s]

Cached text to cache/012_2015_annual_report_text.txt
Total text length: 142136 characters
Language detection: de
Created 12 chunks
Loading translation model for de→en...


Device set to use cpu



Translating 12 chunks from de to English...


Translating:   0%|          | 0/2 [00:00<?, ?it/s]

Your input_length: 512 is bigger than 0.9 * max_length: 512. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)


Result type: <class 'dict'>
Result content: {'translation_text': '5102 thcirebstfähseG Annual Report 2015 At a glance 2014 2015 Change in pig iron purchase in kt *) 2 018 2 060 + 2.1% crude steel production in kt 2 345 2 401 + 2.4% gross plate production total in kt 1 820 1 856 + 2.0% of which in Dillingen in kt 1 258 1 296 + 3.0% of which in Dunkerque in kt 562 560 – 0.4 % total dispatch in kt 2 441 2 451 + 0.4 % of which heavy plate in kt 1 767 1 843 + 4.3% of which semi-finished products in kt 674 608 – 9.8% sales revenues to countries in mio E Germany 640 669 France 347 289 other EU countries 384 396 remaining exports 501 378 total sales revenues 1 872 1 732 – 7.5 % total workforce (excluding apprentices) as at 31.12.'}
Extracted text: 5102 thcirebstfähseG Annual Report 2015 At a glance 2014 2015 Change in pig iron purchase in kt *) 2
Result type: <class 'dict'>
Result content: {'translation_text': 'Annual Report – 24 Environmental Statement – 24 Environmental Statement – 24 Enviro

In [59]:
analyzer.inspect_translation(pdf_path, n_samples=2)

No cached data found for data/reports/Dillinger/012_2015_annual_report.pdf


In [ ]:

# Step 2: Filter for climate content
climate_chunks = analyzer.filter_climate_chunks(chunks)
print(f"\n✅ Step 2 complete: {len(climate_chunks)} climate-related chunks")

# Step 3: Analyze specificity
analyzed_chunks = analyzer.analyze_specificity(climate_chunks)
print(f"\n✅ Step 3 complete: Specificity analysis done")

# Step 4: (Optional) Analyze sentiment
# analyzed_chunks = analyzer.analyze_sentiment(analyzed_chunks)
# print(f"\n✅ Step 4 complete: Sentiment analysis done")

# Step 5: (Optional) Analyze commitments
# analyzed_chunks = analyzer.analyze_commitments(analyzed_chunks)
# print(f"\n✅ Step 5 complete: Commitment analysis done")

# Step 6: Calculate scores
scores = analyzer.calculate_report_score(analyzed_chunks)
print(f"\n✅ Step 6 complete: Overall specificity = {scores['overall_specificity']:.3f}")

# Step 7: Save results
results = analyzer.save_results(
    pdf_path, chunks, lang, cache_suffix, analyzed_chunks, scores
)

# Step 8: Inspect results
print(f"\n{'='*80}")
print("FINAL SUMMARY")
print(f"{'='*80}")
print(f"File: {Path(pdf_path).name}")
print(f"Language: {lang} → {'translated to en' if 'translated' in cache_suffix else 'original'}")
print(f"Total chunks: {len(chunks)}")
print(f"Climate chunks: {len(climate_chunks)}")
print(f"Overall specificity: {scores['overall_specificity']:.3f}")
print(f"\nLabel distribution:")
for label, count in scores['label_distribution'].items():
    pct = count / len(analyzed_chunks) * 100
    print(f"  {label}: {count} ({pct:.1f}%)")

# Inspect specific chunks
# analyzer.inspect_chunks_interactive(pdf_path, mode="high_conf", n=3)

In [47]:
from transformers import pipeline
mt = pipeline("translation", "Helsinki-NLP/opus-mt-de-en")

mt("Dies ist ein Test. Wir prüfen die Übersetzung.")


Device set to use cpu


[{'translation_text': "This is a test. We'll check the translation."}]

In [ ]:
"""
Simplified Climate Report Analyzer with Strategic Caching
- One cache file per PDF (contains text + translations)
- Analysis runs fresh every time (it's fast)
- Easy to inspect original vs translated chunks
"""

import json
from pathlib import Path
from typing import List, Dict, Tuple
from datetime import datetime
import pdfplumber
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline
from tqdm.auto import tqdm
import langid


class ClimateReportAnalyzer:
    MIN_CHUNK_LENGTH = 200
    BATCH_SIZE = 48

    def __init__(self, cache_dir="cache"):
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(exist_ok=True)

        self.device = 0 if torch.cuda.is_available() else -1
        self.models = {}
        self.translators = {}

        self.supported_languages = {
            'de': 'Helsinki-NLP/opus-mt-de-en',
            'fr': 'Helsinki-NLP/opus-mt-fr-en',
            'es': 'Helsinki-NLP/opus-mt-es-en',
            'it': 'Helsinki-NLP/opus-mt-it-en',
            'pt': 'Helsinki-NLP/opus-mt-tc-big-pt-en',
            'nl': 'Helsinki-NLP/opus-mt-nl-en',
            'pl': 'Helsinki-NLP/opus-mt-pl-en',
            'sv': 'Helsinki-NLP/opus-mt-sv-en',
            'da': 'Helsinki-NLP/opus-mt-da-en',
            'fi': 'Helsinki-NLP/opus-mt-fi-en',
            'no': 'Helsinki-NLP/opus-mt-no-en',
            'ru': 'Helsinki-NLP/opus-mt-ru-en',
            'zh': 'Helsinki-NLP/opus-mt-zh-en',
            'ja': 'Helsinki-NLP/opus-mt-ja-en',
            'ko': 'Helsinki-NLP/opus-mt-ko-en',
        }

    def load_model(self, model_name: str, task_name: str):
        """Load and cache a model"""
        if task_name not in self.models:
            print(f"Loading {task_name} model...")
            model = AutoModelForSequenceClassification.from_pretrained(model_name)
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.models[task_name] = pipeline(
                "text-classification",
                model=model,
                tokenizer=tokenizer,
                device=self.device
            )
        return self.models[task_name]

    def extract_text_from_pdf(self, pdf_path: str) -> str:
        """Extract text from PDF (cached)"""
        cache_file = self.cache_dir / f"{Path(pdf_path).stem}_text.txt"

        if cache_file.exists():
            print(f"✓ Loading cached PDF text")
            return cache_file.read_text(encoding='utf-8')

        print(f"Extracting text from PDF...")
        text = ""
        with pdfplumber.open(pdf_path) as pdf:
            for page in tqdm(pdf.pages, desc="Reading pages"):
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"

        cache_file.write_text(text, encoding='utf-8')
        print(f"✓ Cached PDF text to {cache_file.name}")
        return text

    def chunk_text_by_paragraphs(self, text: str, max_tokens=400) -> List[str]:
        """Split text into chunks"""
        MAX_CHARS = max_tokens * 4
        paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]

        chunks = []
        current_chunk = ""

        for para in paragraphs:
            para_size = len(para)
            current_size = len(current_chunk)

            if para_size > MAX_CHARS:
                if current_chunk:
                    chunks.append(current_chunk.strip())
                    current_chunk = ""

                sentences = para.split('. ')
                for sent in sentences:
                    if current_size + len(sent) < MAX_CHARS:
                        current_chunk += sent + ". "
                        current_size = len(current_chunk)
                    else:
                        if current_chunk:
                            chunks.append(current_chunk.strip())
                        current_chunk = sent + ". "
                        current_size = len(current_chunk)

            elif current_size + para_size > MAX_CHARS:
                if current_chunk:
                    chunks.append(current_chunk.strip())
                current_chunk = para

            else:
                current_chunk += ("\n\n" if current_chunk else "") + para

        if current_chunk:
            chunks.append(current_chunk.strip())

        return [c for c in chunks if len(c) > self.MIN_CHUNK_LENGTH]

    def detect_language(self, text: str) -> str:
        """Detect language"""
        try:
            sample = text[:5000]
            lang, _ = langid.classify(sample)
            print(f"✓ Language detected: {lang}")
            return lang
        except Exception as e:
            print(f"Language detection failed: {e}. Defaulting to 'en'")
            return 'en'

    def load_translator(self, lang: str):
        """Load translation model"""
        if lang not in self.supported_languages:
            raise ValueError(f"Translation not supported for: {lang}")

        if lang not in self.translators:
            print(f"Loading translator {lang}→en...")
            model_name = self.supported_languages[lang]
            self.translators[lang] = pipeline(
                "translation",
                model=model_name,
                device=self.device,
                max_length=600  # Allow longer outputs
            )

        return self.translators[lang]

    def translate_chunks(self, chunks: List[str], lang: str,
                        batch_size=8) -> List[str]:
        """Translate chunks to English"""
        translator = self.load_translator(lang)
        translated = []

        print(f"Translating {len(chunks)} chunks from {lang}→en...")

        for i in tqdm(range(0, len(chunks), batch_size), desc="Translating"):
            batch = chunks[i:i+batch_size]

            try:
                # Truncate to ~1800 chars (safe for 512 token limit)
                truncated = [chunk[:1800] for chunk in batch]

                results = translator(
                    truncated,
                    batch_size=batch_size,
                    truncation=True,
                    max_length=600
                )

                for result in results:
                    if isinstance(result, dict):
                        text = result.get('translation_text', '')
                    elif isinstance(result, list) and len(result) > 0:
                        text = result[0].get('translation_text', '')
                    else:
                        text = str(result)

                    translated.append(text)

            except Exception as e:
                print(f"\n⚠ Translation error in batch {i//batch_size}: {e}")
                translated.extend(batch)  # Fallback to original

        return translated

    def prepare_pdf(self, pdf_path: str, translate=True) -> Dict:
        """
        Extract, chunk, and optionally translate PDF
        Returns: Dict with original and translated chunks
        """
        pdf_stem = Path(pdf_path).stem
        cache_file = self.cache_dir / f"{pdf_stem}_prepared.json"

        # Check cache
        if cache_file.exists():
            print(f"✓ Loading cached preparation for {pdf_stem}")
            with open(cache_file, 'r', encoding='utf-8') as f:
                return json.load(f)

        print(f"\n{'='*80}")
        print(f"PREPARING PDF: {Path(pdf_path).name}")
        print(f"{'='*80}\n")

        # Extract text
        text = self.extract_text_from_pdf(pdf_path)
        print(f"✓ Extracted {len(text)} characters")

        # Detect language
        lang = self.detect_language(text)

        # Chunk
        chunks = self.chunk_text_by_paragraphs(text)
        print(f"✓ Created {len(chunks)} chunks")

        # Translate if needed
        chunk_pairs = []
        if lang != 'en' and translate:
            if lang not in self.supported_languages:
                print(f"⚠ Translation not supported for {lang}")
                print(f"  Proceeding with original text")
                translated = chunks
            else:
                translated = self.translate_chunks(chunks, lang)
                print(f"✓ Translated {len(translated)} chunks")

            chunk_pairs = [
                {"original": orig, "translated": trans}
                for orig, trans in zip(chunks, translated)
            ]
        else:
            chunk_pairs = [
                {"original": c, "translated": c}
                for c in chunks
            ]

        # Save
        result = {
            "pdf_path": str(pdf_path),
            "language": lang,
            "num_chunks": len(chunks),
            "chunk_pairs": chunk_pairs,
            "prepared_at": str(datetime.now())
        }

        with open(cache_file, 'w', encoding='utf-8') as f:
            json.dump(result, f, ensure_ascii=False, indent=2)

        print(f"✓ Cached to {cache_file.name}")
        return result

    def inspect_preparation(self, pdf_path: str, n_samples=3):
        """Show original vs translated chunks side-by-side"""
        pdf_stem = Path(pdf_path).stem
        cache_file = self.cache_dir / f"{pdf_stem}_prepared.json"

        if not cache_file.exists():
            print(f"No prepared data found. Run prepare_pdf() first.")
            return

        with open(cache_file, 'r', encoding='utf-8') as f:
            data = json.load(f)

        lang = data['language']
        pairs = data['chunk_pairs']

        print(f"\n{'='*80}")
        print(f"PREPARATION INSPECTION: {Path(pdf_path).name}")
        print(f"Language: {lang}")
        print(f"Total chunks: {len(pairs)}")
        print(f"{'='*80}\n")

        import random
        samples = random.sample(pairs, min(n_samples, len(pairs)))

        for i, pair in enumerate(samples, 1):
            orig = pair['original']
            trans = pair['translated']

            print(f"{'─'*80}")
            print(f"Sample {i}")
            print(f"{'─'*80}")
            print(f"\nORIGINAL ({lang}):")
            print(f"{orig[:400]}{'...' if len(orig) > 400 else ''}")

            if orig != trans:  # Only show if actually translated
                print(f"\nTRANSLATED (en):")
                print(f"{trans[:400]}{'...' if len(trans) > 400 else ''}")

            print()

    def filter_climate_chunks(self, chunks: List[str], batch_size=None) -> List[Dict]:
        """Filter for climate-related chunks"""
        if batch_size is None:
            batch_size = self.BATCH_SIZE

        detector = self.load_model(
            "climatebert/distilroberta-base-climate-detector",
            "detector"
        )

        print(f"\nFiltering {len(chunks)} chunks for climate content...")
        climate_chunks = []

        for i in tqdm(range(0, len(chunks), batch_size), desc="Detecting"):
            batch = chunks[i:i+batch_size]

            try:
                results = detector(batch, truncation=True, max_length=512,
                                 batch_size=batch_size)

                for chunk, result in zip(batch, results):
                    if result['label'].lower() in ['climate', 'yes', 'climate-related']:
                        climate_chunks.append({
                            'text': chunk,
                            'detector_score': result['score']
                        })
            except Exception as e:
                print(f"Error in batch {i}: {e}")
                continue

        print(f"✓ Kept {len(climate_chunks)} climate chunks "
              f"({len(climate_chunks)/len(chunks)*100:.1f}%)")
        return climate_chunks

    def analyze_full_pipeline(self, pdf_path: str, translate=True):
        """Run complete analysis pipeline"""

        # Step 1: Prepare (extract, chunk, translate)
        prepared = self.prepare_pdf(pdf_path, translate=translate)

        # Use translated chunks for analysis
        chunks = [pair['translated'] for pair in prepared['chunk_pairs']]

        print(f"\n{'='*80}")
        print(f"ANALYZING: {Path(pdf_path).name}")
        print(f"{'='*80}")

        # Step 2: Filter climate content
        climate_chunks = self.filter_climate_chunks(chunks)

        # TODO: Add other analysis steps here
        # - analyze_specificity
        # - analyze_sentiment
        # - analyze_commitments

        return {
            'prepared': prepared,
            'climate_chunks': climate_chunks
        }


# Usage example
if __name__ == "__main__":
    analyzer = ClimateReportAnalyzer()

    # Prepare PDF (extract + translate, cached)
    pdf_path = "data/reports/example.pdf"
    prepared = analyzer.prepare_pdf(pdf_path, translate=True)

    # Inspect preparation
    analyzer.inspect_preparation(pdf_path, n_samples=2)

    # Run full analysis (uses prepared data)
    results = analyzer.analyze_full_pipeline(pdf_path, translate=True)

In [64]:
"""
Simplified Climate Report Analyzer with Step-by-Step Pipeline
- Explicit steps: extract → translate → analyze
- Each step can be run independently
- Strategic caching only for slow operations
"""

import json
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from datetime import datetime
import pdfplumber
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline
from tqdm.auto import tqdm
import langid


class ClimateReportAnalyzer:
    MIN_CHUNK_LENGTH = 200
    BATCH_SIZE = 48

    def __init__(self, cache_dir="cache"):
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(exist_ok=True)

        self.device = 0 if torch.cuda.is_available() else -1
        self.models = {}
        self.translators = {}

        self.supported_languages = {
            'de': 'Helsinki-NLP/opus-mt-de-en',
            'fr': 'Helsinki-NLP/opus-mt-fr-en',
            'es': 'Helsinki-NLP/opus-mt-es-en',
            'it': 'Helsinki-NLP/opus-mt-it-en',
            'pt': 'Helsinki-NLP/opus-mt-tc-big-pt-en',
            'nl': 'Helsinki-NLP/opus-mt-nl-en',
            'pl': 'Helsinki-NLP/opus-mt-pl-en',
            'sv': 'Helsinki-NLP/opus-mt-sv-en',
            'da': 'Helsinki-NLP/opus-mt-da-en',
            'fi': 'Helsinki-NLP/opus-mt-fi-en',
            'no': 'Helsinki-NLP/opus-mt-no-en',
            'ru': 'Helsinki-NLP/opus-mt-ru-en',
            'zh': 'Helsinki-NLP/opus-mt-zh-en',
            'ja': 'Helsinki-NLP/opus-mt-ja-en',
            'ko': 'Helsinki-NLP/opus-mt-ko-en',
        }

    # ========================================================================
    # STEP 1: EXTRACT PDF TEXT
    # ========================================================================

    def extract_pdf(self, pdf_path: str) -> Dict:
        """
        Extract text from PDF and chunk it
        Returns: Dict with text, language, and chunks
        """
        pdf_stem = Path(pdf_path).stem
        cache_file = self.cache_dir / f"{pdf_stem}_extracted.json"

        # Check cache
        if cache_file.exists():
            print(f"✓ Loading cached extraction for {pdf_stem}")
            with open(cache_file, 'r', encoding='utf-8') as f:
                return json.load(f)

        print(f"\n{'='*80}")
        print(f"STEP 1: EXTRACTING PDF")
        print(f"{'='*80}\n")

        # Extract text
        text = self._extract_text_from_pdf(pdf_path)
        print(f"✓ Extracted {len(text):,} characters")

        # Detect language
        lang = self._detect_language(text)

        # Chunk text
        chunks = self._chunk_text(text)
        print(f"✓ Created {len(chunks)} chunks")

        # Save
        result = {
            "pdf_path": str(pdf_path),
            "language": lang,
            "num_chunks": len(chunks),
            "chunks": chunks,
            "extracted_at": str(datetime.now())
        }

        with open(cache_file, 'w', encoding='utf-8') as f:
            json.dump(result, f, ensure_ascii=False, indent=2)

        print(f"✓ Cached to {cache_file.name}\n")
        return result

    def _extract_text_from_pdf(self, pdf_path: str) -> str:
        """Extract raw text from PDF pages"""
        text = ""
        with pdfplumber.open(pdf_path) as pdf:
            for page in tqdm(pdf.pages, desc="Reading pages"):
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        return text

    def _detect_language(self, text: str) -> str:
        """Detect language using langid"""
        try:
            sample = text[:5000]
            lang, _ = langid.classify(sample)
            print(f"✓ Language detected: {lang}")
            return lang
        except Exception as e:
            print(f"⚠ Language detection failed: {e}. Defaulting to 'en'")
            return 'en'

    def _chunk_text(self, text: str, max_tokens=400) -> List[str]:
        """Split text into chunks by paragraphs"""
        MAX_CHARS = max_tokens * 4
        paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]

        chunks = []
        current_chunk = ""

        for para in paragraphs:
            para_size = len(para)
            current_size = len(current_chunk)

            # Large paragraph: split by sentences
            if para_size > MAX_CHARS:
                if current_chunk:
                    chunks.append(current_chunk.strip())
                    current_chunk = ""

                sentences = para.split('. ')
                for sent in sentences:
                    if current_size + len(sent) < MAX_CHARS:
                        current_chunk += sent + ". "
                        current_size = len(current_chunk)
                    else:
                        if current_chunk:
                            chunks.append(current_chunk.strip())
                        current_chunk = sent + ". "
                        current_size = len(current_chunk)

            # Would exceed limit: save current and start new
            elif current_size + para_size > MAX_CHARS:
                if current_chunk:
                    chunks.append(current_chunk.strip())
                current_chunk = para

            # Small paragraph: accumulate
            else:
                current_chunk += ("\n\n" if current_chunk else "") + para

        if current_chunk:
            chunks.append(current_chunk.strip())

        # Filter very short chunks
        return [c for c in chunks if len(c) > self.MIN_CHUNK_LENGTH]

    # ========================================================================
    # STEP 2: TRANSLATE (OPTIONAL)
    # ========================================================================

    def translate_pdf(self, pdf_path: str) -> Dict:
        """
        Translate extracted chunks to English
        Returns: Dict with original and translated chunks
        """
        pdf_stem = Path(pdf_path).stem
        cache_file = self.cache_dir / f"{pdf_stem}_translated.json"

        # Check cache
        if cache_file.exists():
            print(f"✓ Loading cached translation for {pdf_stem}")
            with open(cache_file, 'r', encoding='utf-8') as f:
                return json.load(f)

        print(f"\n{'='*80}")
        print(f"STEP 2: TRANSLATING")
        print(f"{'='*80}\n")

        # Load extracted data
        extracted = self.extract_pdf(pdf_path)
        lang = extracted['language']
        chunks = extracted['chunks']

        # Check if translation needed
        if lang == 'en':
            print(f"✓ Already in English, no translation needed")
            result = {
                "pdf_path": str(pdf_path),
                "language": lang,
                "translated": False,
                "chunk_pairs": [
                    {"original": c, "translated": c} for c in chunks
                ],
                "translated_at": str(datetime.now())
            }
        elif lang not in self.supported_languages:
            print(f"⚠ Translation not supported for language: {lang}")
            print(f"  Supported: {list(self.supported_languages.keys())}")
            print(f"  Proceeding with original text\n")
            result = {
                "pdf_path": str(pdf_path),
                "language": lang,
                "translated": False,
                "unsupported": True,
                "chunk_pairs": [
                    {"original": c, "translated": c} for c in chunks
                ],
                "translated_at": str(datetime.now())
            }
        else:
            # Translate
            translated_chunks = self._translate_chunks(chunks, lang)
            print(f"✓ Translated {len(translated_chunks)} chunks")

            # Verify translation worked
            detected = langid.classify(translated_chunks[0][:500])[0]
            print(f"✓ First chunk detected as: {detected}\n")

            result = {
                "pdf_path": str(pdf_path),
                "language": lang,
                "translated": True,
                "chunk_pairs": [
                    {"original": orig, "translated": trans}
                    for orig, trans in zip(chunks, translated_chunks)
                ],
                "translated_at": str(datetime.now())
            }

        # Save
        with open(cache_file, 'w', encoding='utf-8') as f:
            json.dump(result, f, ensure_ascii=False, indent=2)

        print(f"✓ Cached to {cache_file.name}\n")
        return result

    def _load_translator(self, lang: str):
        """Load translation model for given language"""
        if lang not in self.translators:
            print(f"Loading translator {lang}→en...")
            model_name = self.supported_languages[lang]
            self.translators[lang] = pipeline(
                "translation",
                model=model_name,
                device=self.device,
                max_length=600  # Allow longer outputs
            )
        return self.translators[lang]

    def _translate_chunks(self, chunks: List[str], lang: str,
                         batch_size=8) -> List[str]:
        """Translate chunks to English"""
        translator = self._load_translator(lang)
        translated = []

        print(f"Translating {len(chunks)} chunks from {lang}→en...")

        for i in tqdm(range(0, len(chunks), batch_size), desc="Translating"):
            batch = chunks[i:i+batch_size]

            try:
                # Truncate to ~1800 chars (safe for 512 token limit)
                truncated = [chunk[:1800] for chunk in batch]

                results = translator(
                    truncated,
                    batch_size=batch_size,
                    truncation=True,
                    max_length=600
                )

                for result in results:
                    if isinstance(result, dict):
                        text = result.get('translation_text', '')
                    elif isinstance(result, list) and len(result) > 0:
                        text = result[0].get('translation_text', '')
                    else:
                        text = str(result)

                    translated.append(text)

            except Exception as e:
                print(f"\n⚠ Translation error in batch {i//batch_size}: {e}")
                translated.extend(batch)  # Fallback to original

        return translated

    # ========================================================================
    # INSPECTION HELPERS
    # ========================================================================

    def inspect_extraction(self, pdf_path: str, n_samples=3):
        """Show extracted chunks"""
        pdf_stem = Path(pdf_path).stem
        cache_file = self.cache_dir / f"{pdf_stem}_extracted.json"

        if not cache_file.exists():
            print(f"No extracted data found. Run extract_pdf() first.")
            return

        with open(cache_file, 'r', encoding='utf-8') as f:
            data = json.load(f)

        print(f"\n{'='*80}")
        print(f"EXTRACTION INSPECTION: {Path(pdf_path).name}")
        print(f"Language: {data['language']}")
        print(f"Total chunks: {data['num_chunks']}")
        print(f"{'='*80}\n")

        import random
        chunks = data['chunks']
        samples = random.sample(chunks, min(n_samples, len(chunks)))

        for i, chunk in enumerate(samples, 1):
            print(f"{'─'*80}")
            print(f"Chunk {i} ({len(chunk)} chars)")
            print(f"{'─'*80}")
            print(f"{chunk[:400]}{'...' if len(chunk) > 400 else ''}\n")

    def inspect_translation(self, pdf_path: str, n_samples=3):
        """Show original vs translated chunks side-by-side"""
        pdf_stem = Path(pdf_path).stem
        cache_file = self.cache_dir / f"{pdf_stem}_translated.json"

        if not cache_file.exists():
            print(f"No translation data found. Run translate_pdf() first.")
            return

        with open(cache_file, 'r', encoding='utf-8') as f:
            data = json.load(f)

        lang = data['language']
        pairs = data['chunk_pairs']
        was_translated = data.get('translated', False)

        print(f"\n{'='*80}")
        print(f"TRANSLATION INSPECTION: {Path(pdf_path).name}")
        print(f"Language: {lang}")
        print(f"Translated: {was_translated}")
        print(f"Total chunks: {len(pairs)}")
        print(f"{'='*80}\n")

        import random
        samples = random.sample(pairs, min(n_samples, len(pairs)))

        for i, pair in enumerate(samples, 1):
            orig = pair['original']
            trans = pair['translated']

            print(f"{'─'*80}")
            print(f"Sample {i}")
            print(f"{'─'*80}")
            print(f"\nORIGINAL ({lang}):")
            print(f"{orig[:400]}{'...' if len(orig) > 400 else ''}")

            if orig != trans:  # Only show if actually different
                print(f"\nTRANSLATED (en):")
                print(f"{trans[:400]}{'...' if len(trans) > 400 else ''}")

            print()

    # ========================================================================
    # STEP 3: ANALYSIS
    # ========================================================================

    def load_model(self, model_name: str, task_name: str):
        """Load and cache a model"""
        if task_name not in self.models:
            print(f"Loading {task_name} model...")
            model = AutoModelForSequenceClassification.from_pretrained(model_name)
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.models[task_name] = pipeline(
                "text-classification",
                model=model,
                tokenizer=tokenizer,
                device=self.device
            )
        return self.models[task_name]

    def get_chunks_for_analysis(self, pdf_path: str,
                                use_translated: bool = True) -> List[str]:
        """
        Get chunks ready for analysis

        Args:
            pdf_path: Path to PDF
            use_translated: Use translated chunks if available (default True)

        Returns:
            List of text chunks (translated if available and requested)
        """
        pdf_stem = Path(pdf_path).stem
        trans_cache = self.cache_dir / f"{pdf_stem}_translated.json"
        extract_cache = self.cache_dir / f"{pdf_stem}_extracted.json"

        # Try translated first
        if use_translated and trans_cache.exists():
            with open(trans_cache, 'r', encoding='utf-8') as f:
                data = json.load(f)
            return [pair['translated'] for pair in data['chunk_pairs']]

        # Fall back to extracted
        if extract_cache.exists():
            with open(extract_cache, 'r', encoding='utf-8') as f:
                data = json.load(f)
            return data['chunks']

        # Nothing cached, extract now
        extracted = self.extract_pdf(pdf_path)
        return extracted['chunks']

    def filter_climate_chunks(self, chunks: List[str], batch_size=None) -> List[Dict]:
        """Filter for climate-related chunks"""
        if batch_size is None:
            batch_size = self.BATCH_SIZE

        detector = self.load_model(
            "climatebert/distilroberta-base-climate-detector",
            "detector"
        )

        print(f"\nFiltering {len(chunks)} chunks for climate content...")
        climate_chunks = []

        for i in tqdm(range(0, len(chunks), batch_size), desc="Detecting"):
            batch = chunks[i:i+batch_size]

            try:
                results = detector(batch, truncation=True, max_length=512,
                                 batch_size=batch_size)

                for chunk, result in zip(batch, results):
                    if result['label'].lower() in ['climate', 'yes', 'climate-related']:
                        climate_chunks.append({
                            'text': chunk,
                            'detector_score': result['score']
                        })
            except Exception as e:
                print(f"Error in batch {i}: {e}")
                continue

        print(f"✓ Kept {len(climate_chunks)} climate chunks "
              f"({len(climate_chunks)/len(chunks)*100:.1f}%)")
        return climate_chunks


# ============================================================================
# USAGE EXAMPLE
# ============================================================================

if __name__ == "__main__":
    analyzer = ClimateReportAnalyzer()

    pdf_path = "data/reports/Dillinger/012_2015_annual_report.pdf"

    # STEP 1: Extract PDF (always do this)
    extracted = analyzer.extract_pdf(pdf_path)
    analyzer.inspect_extraction(pdf_path, n_samples=2)

    # STEP 2: Translate (comment out if don't want to translate)
    translated = analyzer.translate_pdf(pdf_path)
    analyzer.inspect_translation(pdf_path, n_samples=2)

    # STEP 3: Get chunks for analysis
    chunks = analyzer.get_chunks_for_analysis(pdf_path, use_translated=True)
    print(f"\n✓ Ready for analysis: {len(chunks)} chunks")

    # Verify language
    # import langid
    # detected = langid.classify(chunks[0][:500])[0]
    # print(f"✓ Chunks detected as: {detected}")

    # STEP 4: Filter climate content
    climate_chunks = analyzer.filter_climate_chunks(chunks)

    # Continue with more analysis...

✓ Loading cached extraction for 012_2015_annual_report

EXTRACTION INSPECTION: 012_2015_annual_report.pdf
Language: de
Total chunks: 12

────────────────────────────────────────────────────────────────────────────────
Chunk 1 (4093 chars)
────────────────────────────────────────────────────────────────────────────────
Finanzmittelfonds am Ende der Periode 67 009 79 539
60
Überleitungsrechnung Finanzmittelfonds
in T  31.12.2015 31 12 2014 *) 1 1 2014 *)
Guthaben bei Kreditinstituten 65 852 78 382 143 124
Sonstige Wertpapiere 1 157 1 157 1 157
Finanzmittelfonds 67 009 79 539 144 281
Veränderung Finanzmittelfonds – 12 530 – 64 742
*) Zur besseren Vergleichbarkeit erfolgt die Darstellung der Kapitalflussrechnung u...

────────────────────────────────────────────────────────────────────────────────
Chunk 2 (2817 chars)
────────────────────────────────────────────────────────────────────────────────
5 048 5 081
Personalaufwand in Mio E 317 344
Bilanzsumme in Mio E 2 962 2 943
Anlagevermögen

Device set to use cpu



Filtering 12 chunks for climate content...


Detecting:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Kept 2 climate chunks (16.7%)
